# 📄 ID-VLM — Notebook 01: Data Preparation

**Goal:** Download the MIDV-2020 subset, explore the annotation structure, and convert raw images + annotations into Unsloth-compatible instruction/answer pairs.

**Output:** `train.jsonl`, `val.jsonl`, `test.jsonl` saved to Google Drive.

---

## What this notebook does:
1. Mount Google Drive & clone the project repo
2. Download MIDV-2020 subset (3-4 document types)
3. Explore annotations and image quality
4. Convert to ChatML instruction pairs
5. Split into train/val/test and save

## 1. Setup & Mount Drive

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

# Project directory on Drive
import os
PROJECT_DIR = '/content/drive/MyDrive/id-vlm'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data/raw', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data/processed', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/outputs', exist_ok=True)
print(f'Project directory: {PROJECT_DIR}')

In [ ]:
# Clone the project repo (or upload src/ and config.py manually)
# Option 1: Clone from GitHub (uncomment and update URL)
# !git clone https://github.com/OmTilwar/ID-VLM.git /content/id-vlm

# Option 2: Upload files manually — upload src/, config.py to /content/id-vlm/
# For now, let's create the necessary files inline

REPO_DIR = '/content/id-vlm'
os.makedirs(REPO_DIR, exist_ok=True)

# If you uploaded the repo, verify the structure
if os.path.exists(f'{REPO_DIR}/config.py'):
    print('✅ Project files found!')
    !ls {REPO_DIR}/src/
else:
    print('⚠️ Project files not found. Upload src/ and config.py to /content/id-vlm/')
    print('   Or clone your GitHub repo in the cell above.')

In [ ]:
# Install dependencies
!pip install Pillow python-Levenshtein numpy tqdm -q

import sys
sys.path.insert(0, REPO_DIR)

## 2. Download MIDV-2020 Subset

MIDV-2020 is ~124GB total. We only need **3-4 document types** (~300 images).

**Option A: INRIA Rectified Photos** (easiest, ~1-2GB)  
Pre-processed, perspective-corrected photos. No video frames.

**Option B: Official SFTP Download** (richer, requires access request)  
Full dataset with scans, photos, AND video frames.

We'll try Option A first, then show how to adapt for Option B.

In [ ]:
# ── Option A: Download from INRIA GitLab (rectified photos) ──
# This gives us perspective-corrected photos with annotations

RAW_DIR = f'{PROJECT_DIR}/data/raw'

# Selected document types (mix of ID cards and passports)
DOC_TYPES = ['alb_id', 'aze_passport', 'esp_id', 'grc_passport']

print('Downloading MIDV-2020 subset...')
print('Selected document types:', DOC_TYPES)
print()

# Try cloning the rectified photos repo
MIDV_REPO = 'https://gitlab.inria.fr/tneittho/midv2020-rectified-photo.git'

if not os.path.exists(f'{RAW_DIR}/midv2020-rectified-photo'):
    print(f'Cloning from {MIDV_REPO}...')
    !git clone --depth 1 {MIDV_REPO} {RAW_DIR}/midv2020-rectified-photo
else:
    print('Already downloaded.')

# List what we got
!ls {RAW_DIR}/midv2020-rectified-photo/ 2>/dev/null || echo 'Download failed — see manual instructions below'

In [ ]:
# ── Fallback: Manual Download Instructions ──
# If the git clone above fails, you can:
#
# 1. Go to https://arxiv.org/abs/2107.00396 (MIDV-2020 paper)
# 2. Follow the dataset access link
# 3. Download only these folders: alb_id, aze_passport, esp_id, grc_passport
# 4. Upload to Google Drive at: id-vlm/data/raw/<doc_type>/
#
# Expected structure per doc_type:
#   <doc_type>/
#     images/ or photos/
#       <doc_id>/
#         <frame>.jpg
#     ground_truth/ or annotations/
#       <annotations>.json

print('Expected data structure:')
print(f'  {RAW_DIR}/')
for dt in DOC_TYPES:
    print(f'    {dt}/')
    print(f'      images/')
    print(f'      ground_truth/')

## 3. Explore the Dataset

Let's understand what we're working with before converting.

In [ ]:
import glob
import json
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

# Survey the downloaded data
data_root = f'{RAW_DIR}/midv2020-rectified-photo'
if not os.path.exists(data_root):
    data_root = RAW_DIR  # Fallback to manual download location

print(f'Data root: {data_root}')
print()

for dt in DOC_TYPES:
    dt_dir = os.path.join(data_root, dt)
    if os.path.exists(dt_dir):
        # Count images
        images = glob.glob(os.path.join(dt_dir, '**', '*.jpg'), recursive=True)
        images += glob.glob(os.path.join(dt_dir, '**', '*.png'), recursive=True)
        images += glob.glob(os.path.join(dt_dir, '**', '*.tif'), recursive=True)
        
        # Count annotations
        jsons = glob.glob(os.path.join(dt_dir, '**', '*.json'), recursive=True)
        
        print(f'{dt}: {len(images)} images, {len(jsons)} JSON files')
    else:
        print(f'{dt}: ❌ Directory not found at {dt_dir}')

In [ ]:
# Inspect annotation structure
# Find the first JSON annotation file and examine it

sample_jsons = glob.glob(os.path.join(data_root, '**', '*.json'), recursive=True)

if sample_jsons:
    sample_json = sample_jsons[0]
    print(f'Sample annotation file: {sample_json}')
    print()
    
    with open(sample_json, 'r') as f:
        data = json.load(f)
    
    # Pretty-print the structure (first 2000 chars)
    formatted = json.dumps(data, indent=2, ensure_ascii=False)
    print(formatted[:2000])
    if len(formatted) > 2000:
        print(f'\n... ({len(formatted)} total chars)')
else:
    print('No JSON annotation files found. Check your data directory.')

In [ ]:
# Visualize a few sample documents

sample_images = glob.glob(os.path.join(data_root, '**', '*.jpg'), recursive=True)[:8]

if sample_images:
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.flatten()
    
    for i, img_path in enumerate(sample_images):
        img = Image.open(img_path)
        axes[i].imshow(img)
        axes[i].set_title(f'{Path(img_path).parent.name}/{Path(img_path).name}\n{img.size}', fontsize=8)
        axes[i].axis('off')
    
    # Hide unused axes
    for j in range(len(sample_images), len(axes)):
        axes[j].axis('off')
    
    plt.suptitle('MIDV-2020 Sample Documents', fontsize=14)
    plt.tight_layout()
    plt.savefig(f'{PROJECT_DIR}/outputs/sample_documents.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved to {PROJECT_DIR}/outputs/sample_documents.png')
else:
    print('No images found.')

## 4. Understand the Annotation Format

Before running the data pipeline, let's understand exactly how MIDV-2020 stores ground truth so we can adapt the parser if needed.

In [ ]:
# Catalog all unique field names across the dataset
all_field_names = set()
field_count_by_type = {}

for jf in glob.glob(os.path.join(data_root, '**', '*.json'), recursive=True):
    try:
        with open(jf, 'r') as f:
            data = json.load(f)
        
        # Infer doc type from path
        doc_type = 'unknown'
        for dt in DOC_TYPES:
            if dt in jf.lower():
                doc_type = dt
                break
        
        # Extract field names (adapt based on actual structure)
        # Strategy 1: VIA format
        if '_via_img_metadata' in data:
            for key, entry in data['_via_img_metadata'].items():
                for region in entry.get('regions', []):
                    fname = region.get('region_attributes', {}).get('field_name', '')
                    if fname:
                        all_field_names.add(fname)
                        if doc_type not in field_count_by_type:
                            field_count_by_type[doc_type] = set()
                        field_count_by_type[doc_type].add(fname)
        
        # Strategy 2: Direct field dict
        elif isinstance(data, dict):
            for key, val in data.items():
                if isinstance(val, dict):
                    for field_name in val.keys():
                        all_field_names.add(field_name)
                        if doc_type not in field_count_by_type:
                            field_count_by_type[doc_type] = set()
                        field_count_by_type[doc_type].add(field_name)
    except Exception as e:
        pass

print('All unique field names found:')
for fn in sorted(all_field_names):
    print(f'  - {fn}')

print(f'\nFields by document type:')
for dt, fields in field_count_by_type.items():
    print(f'  {dt}: {sorted(fields)}')

## 5. Convert to Instruction Pairs

Now we run the data pipeline from `src/dataset.py` to convert raw annotations into ChatML instruction pairs.

In [ ]:
# Import the data pipeline
from src.dataset import (
    create_dataset, split_dataset, save_dataset,
    build_instruction_pair, get_capture_mode,
    parse_midv_annotation, parse_midv_ground_truth
)
import config

# Override paths for Colab
config.RAW_DATA_DIR = data_root
config.PROCESSED_DATA_DIR = f'{PROJECT_DIR}/data/processed'

# Create dataset
print('Creating instruction pairs from MIDV-2020...')
dataset = create_dataset(
    data_dir=data_root,
    doc_types=DOC_TYPES,
)

print(f'\nTotal instruction pairs: {len(dataset)}')

In [ ]:
# If the automatic parser didn't find data, we can manually build pairs
# This cell handles alternate MIDV-2020 directory layouts

if len(dataset) == 0:
    print('Automatic parsing found 0 samples. Building pairs manually...')
    print()
    
    dataset = []
    
    for dt in DOC_TYPES:
        dt_dir = os.path.join(data_root, dt)
        if not os.path.exists(dt_dir):
            continue
        
        # Find all JSON files for this doc type
        json_files = glob.glob(os.path.join(dt_dir, '**', '*.json'), recursive=True)
        
        for jf in json_files:
            try:
                with open(jf, 'r') as f:
                    data = json.load(f)
                
                # Extract ground truth fields
                if isinstance(data, dict):
                    for doc_id, fields in data.items():
                        if not isinstance(fields, dict):
                            continue
                        
                        # Find corresponding image
                        img_patterns = [
                            os.path.join(dt_dir, '**', f'{doc_id}*.jpg'),
                            os.path.join(dt_dir, '**', f'{doc_id}*.png'),
                            os.path.join(dt_dir, '**', f'*{doc_id}*.jpg'),
                        ]
                        
                        for pattern in img_patterns:
                            matches = glob.glob(pattern, recursive=True)
                            if matches:
                                # Use first matching image
                                img_path = matches[0]
                                pair = build_instruction_pair(
                                    image_path=img_path,
                                    fields={k.lower().replace(' ', '_'): str(v) for k, v in fields.items()},
                                    doc_type=dt,
                                )
                                dataset.append(pair)
                                break
            except Exception as e:
                pass
    
    print(f'Manual parsing found {len(dataset)} instruction pairs')
else:
    print(f'✅ Automatic parsing found {len(dataset)} instruction pairs')

In [ ]:
# Inspect a few samples
print('=' * 60)
print('SAMPLE INSTRUCTION PAIRS')
print('=' * 60)

for i, sample in enumerate(dataset[:3]):
    print(f'\n--- Sample {i+1} ---')
    metadata = sample.get('metadata', {})
    print(f'Doc type:     {metadata.get("doc_type", "unknown")}')
    print(f'Capture mode: {metadata.get("capture_mode", "unknown")}')
    print(f'Image:        {metadata.get("image_path", "N/A")}')
    print(f'Num fields:   {metadata.get("num_fields", 0)}')
    
    # Show the prompt
    user_msg = sample['messages'][0]
    for c in user_msg['content']:
        if c['type'] == 'text':
            print(f'Prompt:       {c["text"][:80]}...')
    
    # Show the expected answer
    assistant_msg = sample['messages'][1]
    answer_text = assistant_msg['content'][0]['text']
    print(f'Answer:       {answer_text}')

In [ ]:
# Dataset statistics
from collections import Counter

doc_types_dist = Counter(s.get('metadata', {}).get('doc_type', 'unknown') for s in dataset)
capture_modes_dist = Counter(s.get('metadata', {}).get('capture_mode', 'unknown') for s in dataset)

print('Distribution by document type:')
for dt, count in doc_types_dist.most_common():
    print(f'  {dt}: {count} ({100*count/len(dataset):.1f}%)')

print(f'\nDistribution by capture mode:')
for cm, count in capture_modes_dist.most_common():
    print(f'  {cm}: {count} ({100*count/len(dataset):.1f}%)')

# Field count distribution
field_counts = [s.get('metadata', {}).get('num_fields', 0) for s in dataset]
print(f'\nFields per document: min={min(field_counts)}, max={max(field_counts)}, mean={np.mean(field_counts):.1f}')

## 6. Split & Save

In [ ]:
# Split into train/val/test with stratification by doc_type
train, val, test = split_dataset(dataset, 0.7, 0.15, 0.15)

print(f'Train: {len(train)} samples')
print(f'Val:   {len(val)} samples')
print(f'Test:  {len(test)} samples')

# Verify stratification
print('\nTrain doc type distribution:')
for dt, count in Counter(s['metadata']['doc_type'] for s in train).most_common():
    print(f'  {dt}: {count}')

print('\nTest doc type distribution:')
for dt, count in Counter(s['metadata']['doc_type'] for s in test).most_common():
    print(f'  {dt}: {count}')

In [ ]:
# Save to Google Drive
output_dir = f'{PROJECT_DIR}/data/processed'
os.makedirs(output_dir, exist_ok=True)

save_dataset(train, f'{output_dir}/train.jsonl')
save_dataset(val, f'{output_dir}/val.jsonl')
save_dataset(test, f'{output_dir}/test.jsonl')
save_dataset(dataset, f'{output_dir}/full.jsonl')

print(f'\n✅ Data saved to {output_dir}/')
!ls -la {output_dir}/

## 7. Quick Sanity Check

Load a saved file and verify it round-trips correctly.

In [ ]:
from src.dataset import load_dataset

# Verify roundtrip
loaded_train = load_dataset(f'{output_dir}/train.jsonl')

print(f'Loaded {len(loaded_train)} training samples')
print(f'First sample doc_type: {loaded_train[0]["metadata"]["doc_type"]}')
print(f'First sample answer: {loaded_train[0]["messages"][1]["content"][0]["text"][:100]}')
print()
print('✅ Data pipeline verified! Proceed to Notebook 02 for baseline evaluation.')